In [1]:
import os
import re
from pathlib import Path
import wikipediaapi
import json
from typing import List
import faiss
from torch.autograd import set_multithreading_enabled

In [2]:
USER_AGENT = os.getenv('Learning project: TourGuideAI/1.0', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
                          ' Chrome/91.0.4472.124 Safari/537.36')

In [3]:
URL_PATH_WIKI = Path('../data/wiki')
def load_urls(path: Path):
    urls = []
    for file in path.glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry["url"])
    return urls
urls_wiki = load_urls(URL_PATH_WIKI)
urls_wiki

['https://de.wikipedia.org/wiki/Spreewald',
 'https://de.wikipedia.org/wiki/L%C3%BCbbenau/Spreewald',
 'https://de.wikipedia.org/wiki/Brandenburg_an_der_Havel',
 'https://de.wikipedia.org/wiki/Potsdam',
 'https://de.wikipedia.org/wiki/Potsdamer_Stadtschloss',
 'https://de.wikipedia.org/wiki/Alter_Markt_(Potsdam)',
 'https://de.wikipedia.org/wiki/Gro%C3%9Fer_Spreewaldhafen_L%C3%BCbbenau',
 'https://de.wikipedia.org/wiki/Kahn',
 'https://de.wikipedia.org/wiki/Landkreis_Oberspreewald-Lausitz',
 'https://de.wikipedia.org/wiki/Raddusch',
 'https://de.wikipedia.org/wiki/Vetschau/Spreewald',
 'https://de.wikipedia.org/wiki/Sorbisches_Siedlungsgebiet',
 'https://de.wikipedia.org/wiki/Slawenburg_Raddusch',
 'https://de.wikipedia.org/wiki/Bahnhof_Raddusch',
 'https://de.wikipedia.org/wiki/L%C3%BCbben_(Spreewald)',
 'https://de.wikipedia.org/wiki/Datei:Bootshaus_Kaupen_-_Paddelboot_im_Spreewald_-_Quodda.jpg',
 '',
 '']

In [4]:
wiki = wikipediaapi.Wikipedia(language='de',
                             extract_format=wikipediaapi.ExtractFormat.WIKI,
                             user_agent=USER_AGENT)

def load_wikipedia_text(title: str) -> dict:
    page = wiki.page(title)
    if not page.exists():
        return {'error': 'Page not found'}
    text = page.text
    return {
        'text': text,
        'title': page.title,
        'source': page.fullurl,
        'license': 'CC BY-SA 4.0'
    }

#Titel aus URL extrahieren
def extract_title_from_url(url: str) -> str:
    match = re.search(r'/wiki/([^#]+)', url)
    if match:
        return match.group(1)
    return None

def get_summary_from_wiki(text:str, n=2) -> str:
    paragraphs = text.split('\n\n')
    summary = ' '.join(paragraphs[:n])
    return summary

wiki_data = {}
for url in urls_wiki:
    title = extract_title_from_url(url)
    wiki_data[title] = load_wikipedia_text(title)


# Ergebnis als JSON ausgeben

#print(json.dumps(wiki_data, indent=2, ensure_ascii=False))
print(f"\n {len(wiki_data)} Wikipedia-Artikel geladen.")
for i, (title, data) in enumerate(wiki_data.items(),start=1):
    if 'text' in data:
        summary = get_summary_from_wiki(data['text'])
        print(f"\n--- Artikel {i}: ---")
        print(f"\n {data['title']}\n{summary}\n")


 17 Wikipedia-Artikel geladen.

--- Artikel 1: ---

 Spreewald
Der Spreewald (niedersorbisch Błota, „die Sümpfe“) ist ein ausgedehntes Niederungsgebiet und eine historische Kulturlandschaft im Südosten Brandenburgs. Hauptmerkmal ist die natürliche Flusslaufverzweigung der Spree, die durch angelegte Kanäle deutlich erweitert wurde. Als Auen- und Moorlandschaft besitzt sie für den Naturschutz überregionale Bedeutung und ist als Biosphärenreservat geschützt (siehe Biosphärenreservat Spreewald). Der Spreewald als Kulturlandschaft wurde entscheidend durch die Sorben geprägt. Das Gebiet ist eines der bekanntesten und beliebtesten Reiseziele im Land Brandenburg. Insgesamt 222,8 Kilometer im Oberspreewald und 45,4 Kilometer im Unterspreewald sind als Landeswasserstraße klassifiziert. Geografie
Gliederung
Der Spreewald befindet sich in den Landkreisen Spree-Neiße, Dahme-Spreewald und Oberspreewald-Lausitz. Er wird in den südlichen und größeren Oberspreewald und den nördlichen, kleineren Unters

In [5]:
def clean_text(text):
     # Kleinbuchstaben, Entfernen von Sonderzeichen
    text = text.lower().replace(",", " ").replace("!", " ").replace("?", " ").replace("-", " ").strip()

    # text in Liste von Wörtern umwandeln
    text = text.split()

    return text

for title, data in wiki_data.items():
        if 'text' in data:
            print(clean_text(data['text']))
        else:                     # Fehler abfangen, sonst Error
            print(f"Fehler bei {title}: {data.get('error', 'Unbekannter Fehler')}")

['der', 'spreewald', '(niedersorbisch', 'błota', '„die', 'sümpfe“)', 'ist', 'ein', 'ausgedehntes', 'niederungsgebiet', 'und', 'eine', 'historische', 'kulturlandschaft', 'im', 'südosten', 'brandenburgs.', 'hauptmerkmal', 'ist', 'die', 'natürliche', 'flusslaufverzweigung', 'der', 'spree', 'die', 'durch', 'angelegte', 'kanäle', 'deutlich', 'erweitert', 'wurde.', 'als', 'auen', 'und', 'moorlandschaft', 'besitzt', 'sie', 'für', 'den', 'naturschutz', 'überregionale', 'bedeutung', 'und', 'ist', 'als', 'biosphärenreservat', 'geschützt', '(siehe', 'biosphärenreservat', 'spreewald).', 'der', 'spreewald', 'als', 'kulturlandschaft', 'wurde', 'entscheidend', 'durch', 'die', 'sorben', 'geprägt.', 'das', 'gebiet', 'ist', 'eines', 'der', 'bekanntesten', 'und', 'beliebtesten', 'reiseziele', 'im', 'land', 'brandenburg.', 'insgesamt', '222', '8', 'kilometer', 'im', 'oberspreewald', 'und', '45', '4', 'kilometer', 'im', 'unterspreewald', 'sind', 'als', 'landeswasserstraße', 'klassifiziert.', 'geografie', '

In [6]:
def chunk_text(text: str, max_words: int = 200, overlap: int = 50) -> List[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap  # Überlappung für Kontext
        if start < 0:
            start = 0

    return chunks

chunks = []

for data in wiki_data.values():
    if 'text' in data:
        chunks.extend(chunk_text(data['text'], max_words=200, overlap=50))
print(f"{len(chunks)} Text-Chunks aus Wikipedia-Artikeln erstellt.")

287 Text-Chunks aus Wikipedia-Artikeln erstellt.


In [7]:
#Chunking mit metadaten
wiki_chunks = []
for title, data in wiki_data.items():
    if 'text' in data:
        chunks = chunk_text(data['text'], max_words=200, overlap=50)
        for i, chunk in enumerate(chunks):
            wiki_chunks.append({
                "text": chunk,
                "title": data['title'],
                "url": data['source']
            })
print(f"{len(wiki_chunks)} Text-Chunks mit Metadaten erstellt.")

287 Text-Chunks mit Metadaten erstellt.


In [8]:
#Embeddings für Wiki-Chunks erstellen
from sentence_transformers import SentenceTransformer
model = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model)
wiki_embeddings = embedding_model.encode([chunk['text'] for chunk in wiki_chunks])
print(f"Embeddings für {len(wiki_embeddings)} Wiki-Chunks erstellt.")


Embeddings für 287 Wiki-Chunks erstellt.


In [9]:
#FAISS-Index erstellen

dimension = wiki_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(wiki_embeddings)
print("Anzahl Vektoren im Index:", index.ntotal)

Anzahl Vektoren im Index: 287


In [10]:
#TestSuche
def search(query: str, k: int = 5):
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    D, I = index.search(q_emb, k)  # D: Scores, I: Indizes

    results = []
    for score, idx in zip(D[0], I[0]):
        chunk = wiki_chunks[int(idx)]
        results.append({
            "title": chunk['title'],
            "url": chunk['url'],
            "text": chunk['text'],
            "score": score
        })
    return results





In [11]:
query = "Wo kann man in Brandenburg gut Kanufahren, besonders für Anfänger?"
results = search(query, k=5)

for r in results:
    print(f"\n--- Treffer: {r['title']} ---")
    print(f"Score: {r['score']:.4f}")
    print(f"URL: {r['url']}")
    print(f"Text: {r['text'][:300]}...")  # Nur die ersten 300 Zeichen anzeigen





--- Treffer: Brandenburg an der Havel ---
Score: 0.9265
URL: https://de.wikipedia.org/wiki/Brandenburg_an_der_Havel
Text: (= Landschaften in Deutschland. Werte der deutschen Heimat. Band 69). Böhlau Verlag, Köln 2006, ISBN 3-412-09103-0. Wolfgang Kusior (Bearb.): Chronik der Stadt Brandenburg. Hrsg. Arbeitskreis Stadtgeschichte im Brandenburgischen Kulturbund e. V. Neddermeyer, Berlin 2003, ISBN 3-933254-40-X. Wolfgang...

--- Treffer: Brandenburg an der Havel ---
Score: 0.9418
URL: https://de.wikipedia.org/wiki/Brandenburg_an_der_Havel
Text: 1: Dominsel – Altstadt – Neustadt. (= Denkmaltopographie Bundesrepublik Deutschland. Denkmale in Brandenburg. Band 1.1). Wernersche Verlagsgesellschaft, Worms am Rhein 1994, ISBN 3-88462-105-X. Udo Geiseler, Klaus Heß (Hrsg.): Brandenburg an der Havel. Lexikon zur Stadtgeschichte (= Einzelveröffentl...

--- Treffer: Brandenburg an der Havel ---
Score: 0.9638
URL: https://de.wikipedia.org/wiki/Brandenburg_an_der_Havel
Text: in der Gemarkung Brande